# 01 Prepare and embed

Kaggle GPU notebook. Estimated runtime: 10-30 min. Run cells in order. Outputs are written under `/kaggle/working/protfunc`; completed embedding shards are skipped on rerun.

In [ ]:
from pathlib import Path
import random,json,subprocess,shutil
import numpy as np
SEED=42
TOP_K=200
MODEL_NAME="facebook/esm2_t6_8M_UR50D"
DATA_DIR = "/kaggle/input/competitions/cafa-5-protein-function-prediction"
FASTA=f"{DATA_DIR}/Train/train_sequences.fasta"
TERMS=f"{DATA_DIR}/Train/train_terms.tsv"
OBO=f"{DATA_DIR}/Train/go-basic.obo"
TAXONOMY=f"{DATA_DIR}/Train/train_taxonomy.tsv"
IA_FILE=f"{DATA_DIR}/IA.txt"
WORK=Path("/kaggle/working/protfunc")
random.seed(SEED); np.random.seed(SEED)
missing=[p for p in [FASTA,TERMS,OBO,TAXONOMY] if not Path(p).is_file()]
if missing: raise FileNotFoundError("Missing CAFA inputs: "+", ".join(missing))
if not Path(IA_FILE).is_file(): print("Optional IA.txt unavailable; S-min is outside this reduced scope.")
WORK.mkdir(parents=True,exist_ok=True)
print("All required CAFA input files exist")

## Install dependencies and check MMseqs2

In [ ]:
!pip -q install biopython
MMSEQS_OK=shutil.which("mmseqs") is not None
if not MMSEQS_OK:
    try:
        subprocess.run(["apt-get","update","-qq"],check=True)
        subprocess.run(["apt-get","install","-y","-qq","mmseqs2"],check=True)
        MMSEQS_OK=shutil.which("mmseqs") is not None
    except Exception as e: print("MMseqs2 install failed; random split fallback:",e)
print("MMseqs2 available:",MMSEQS_OK)

## Subsample, propagate labels, choose top 200 per aspect

In [ ]:
from Bio import SeqIO
from collections import defaultdict,Counter
seqs={r.id:str(r.seq).upper() for r in SeqIO.parse(FASTA,"fasta") if len(r.seq)<=512}
ids=sorted(seqs); random.shuffle(ids); ids=ids[:40000]; seqs={k:seqs[k] for k in ids}
ann=defaultdict(set)
for row in open(TERMS):
 c=row.rstrip().split("\t")
 if len(c)>=2 and c[0] in seqs: ann[c[0]].add(c[1])
parents=defaultdict(set); aspect={}; term=None; ns={"molecular_function":"MF","biological_process":"BP","cellular_component":"CC"}
for raw in open(OBO):
 line=raw.strip()
 if line=="[Term]": term=None
 elif line.startswith("id: GO:"): term=line[4:]
 elif term and line.startswith("is_a: GO:"): parents[term].add(line.split()[1])
 elif term and line.startswith("namespace:"): aspect[term]=ns.get(line.split(":",1)[1].strip())
labels={}
for p,ts in ann.items():
 x=set(ts); todo=list(ts)
 while todo:
  for q in parents[todo.pop()]:
   if q not in x: x.add(q); todo.append(q)
 labels[p]=x
counts=Counter(t for ts in labels.values() for t in ts)
term_lists={a:[t for t,n in sorted(((t,n) for t,n in counts.items() if aspect.get(t)==a and t not in {"GO:0003674","GO:0008150","GO:0005575"}),key=lambda z:(-z[1],z[0]))[:TOP_K]] for a in ("MF","BP","CC")}
print(len(seqs),{a:len(v) for a,v in term_lists.items()})
json.dump(term_lists,open(WORK/"term_lists.json","w"),indent=2)
json.dump({p:sorted(ts) for p,ts in labels.items()},open(WORK/"labels.json","w"),indent=2)


## Cluster at 30% identity; assign whole clusters; record fallback

In [ ]:
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
fasta=WORK/"subset.fasta"; SeqIO.write([SeqRecord(Seq(v),id=k,description="") for k,v in seqs.items()],fasta,"fasta")
meta={"seed":SEED,"method":"random_fallback"}; groups={}
if MMSEQS_OK:
 try:
  prefix=str(WORK/"cluster"); subprocess.run(["mmseqs","easy-cluster",str(fasta),prefix,str(WORK/"tmp"),"--min-seq-id","0.3","-c","0.8"],check=True)
  groups=defaultdict(list)
  for row in open(prefix+"_cluster.tsv"):
   rep,member=row.rstrip().split("\t"); groups[rep].append(member)
  meta["method"]="mmseqs_easy_cluster"
 except Exception as e: print("MMseqs failed; random fallback:",e)
if not groups: groups={k:[k] for k in seqs}
gs=list(groups.values()); random.Random(SEED).shuffle(gs); n=sum(map(len,gs)); seen=0; splits={"train":[],"val":[],"test":[]}
for g in gs:
 s="train" if seen<.8*n else "val" if seen<.9*n else "test"; splits[s]+=g; seen+=len(g)
meta["sizes"]={k:len(v) for k,v in splits.items()}; json.dump({"splits":splits,"metadata":meta},open(WORK/"splits.json","w"),indent=2)

## Extract frozen ESM-2 mean pooled embeddings in resumable shards

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModel
assert torch.cuda.is_available(),"Enable Kaggle GPU"
tok=AutoTokenizer.from_pretrained(MODEL_NAME); model=AutoModel.from_pretrained(MODEL_NAME).cuda().eval(); ed=WORK/"embeddings"; ed.mkdir(exist_ok=True); all_ids=list(seqs)
for lo in range(0,len(all_ids),1000):
 out=ed/f"emb_{lo:06d}.npy"; idfile=ed/f"ids_{lo:06d}.json"
 if out.exists() and idfile.exists(): continue
 keys=all_ids[lo:lo+1000]; chunks=[]
 with torch.no_grad():
  for b in range(0,len(keys),16):
   batch=tok([seqs[k] for k in keys[b:b+16]],return_tensors="pt",padding=True,truncation=True,max_length=514,return_special_tokens_mask=True).to("cuda"); special=batch.pop("special_tokens_mask"); h=model(**batch).last_hidden_state; mask=(batch["attention_mask"]*(1-special)).unsqueeze(-1); chunks.append(((h*mask).sum(1)/mask.sum(1).clamp(min=1)).cpu().numpy().astype(np.float16))
 np.save(out,np.concatenate(chunks)); json.dump(keys,open(idfile,"w"))
print("Shards saved to",ed)
json.dump({"model_name":MODEL_NAME,"seed":SEED},open(WORK/"embedding_config.json","w"),indent=2)
